In [1]:
import torch
import torch.nn as nn 
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from torch.optim import Adam
import pandas as pd 
import numpy as np 

import re

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(
    log_dir="runs/LSTM"
)


from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import os

print(os.getcwd(), device)


[nltk_data] Downloading package punkt to /home/eshaan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/eshaan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/eshaan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


/home/eshaan/ML-CODE cuda


In [2]:
from Word2Vec.main import simpleW2V, encoder, most_similar, embedded_sentence, get_weight_vector

encoder = encoder('/home/eshaan/ML-CODE/datasets/hp_books',5000)
word_idx_dict, encoded_chunks, idx_word_dict = encoder.chunk_encoder(chunk_size=50,
                                                                    remove_stop_words= True, 
                                                                    encode=True, 
                                                                    pad_last=True)
word_idx_dict, word_chunks, idx_word_dict = encoder.chunk_encoder(chunk_size=50,
                                                                    remove_stop_words= True, 
                                                                    encode=False, 
                                                                    pad_last=True)

# word_idx_dict, encoded_sentences, idx_word_dict = encoder.sentence_encoder(True)


[nltk_data] Downloading package punkt to /home/eshaan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/eshaan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/eshaan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:

vocab_size = 5000
feature_count = 50

checkpoint = torch.load(r'/home/eshaan/ML-CODE/Word2Vec/sentence-tokenised.pth')

w2v_model = simpleW2V(vocab_size,feature_count)
w2v_model.load_state_dict(checkpoint['model'])
w2v_model.state_dict()

embedding_w = w2v_model.output.weight

# most_similar(w2v_model, "harry", word_idx_dict,"out",20)

In [ ]:
class simpleLSTM(nn.Module):

    def __init__(self, embedding_w, vocab_size, embedding_dim, hidden_dim, device, freeze_embedding):
        super().__init__()
            
        if not device:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = device

        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(
            num_embeddings= vocab_size,
            embedding_dim=embedding_dim
        )
        with torch.no_grad() :
            self.embedding.weight.data.copy_(embedding_w)

        if freeze_embedding :
            self.embedding.requires_grad_(False)
        else:
            self.embedding.requires_grad_(True)


        self.F_t = nn.Sequential(
            nn.Linear(
                in_features = embedding_dim + hidden_dim,
                out_features = hidden_dim
            ),
            nn.Sigmoid()
        )

        self.I_t = nn.Sequential(
            nn.Linear(
                in_features = embedding_dim + hidden_dim,
                out_features = hidden_dim
            ),
            nn.Sigmoid()
        )

        self.C_bar_t = nn.Sequential(
            nn.Linear(
                in_features = embedding_dim + hidden_dim,
                out_features = hidden_dim
            ),
            nn.Tanh()
        )

        self.O_t = nn.Sequential(
            nn.Linear(
                in_features = embedding_dim + hidden_dim,
                out_features = hidden_dim
            ),
            nn.Sigmoid()
        )

        self.output = nn.Linear(
            in_features= hidden_dim,
            out_features= vocab_size
        )
        
    def forward(self, encoded_input_sequence):
        # input_dim = (batch, Timesteps)

        batch_size,timesteps = encoded_input_sequence.shape
        # if not LTM_context :
        LTM_context = torch.zeros(size=(batch_size, self.hidden_dim), device=self.device)
        # if not STM_context :
        STM_context = torch.zeros(size=(batch_size, self.hidden_dim), device=self.device)


        embedded_input_sequence = self.embedding(encoded_input_sequence) #shape = (batch, Timesteps, embedding_dim)
        logits_per_timestep = []
        for timestep in range(timesteps) : 
            token = embedded_input_sequence[: ,timestep,:]
            F_t = self.F_t(torch.cat([STM_context, token], dim=1))
            LTM_context = F_t * LTM_context

            I_t = self.I_t(torch.cat([STM_context, token], dim=1))
            C_bar_t = self.C_bar_t(torch.cat([STM_context, token], dim=1))
            LTM_context = LTM_context + ( I_t * C_bar_t)

            O_t = self.O_t(torch.cat([STM_context, token], dim=1))
            STM_context = torch.tanh(LTM_context) * O_t
            logits_per_timestep.append(self.output(STM_context))
        logits = torch.stack(logits_per_timestep, dim=1)
        return logits


input                       (B, T)

embedding
                            (B, T, E)

for timestep t:

token                       (B, E)
STM_context                 (B, H)
        ↓ concatenate
combined                    (B, E+H)

F_t                         (B, H)
I_t                         (B, H)
C_bar_t                     (B, H)
O_t                         (B, H)

LTM_context                 (B, H)
STM_context                 (B, H)

Linear(H → V)
logits_t                    (B, V)

after all T timesteps:

logits                      (B, T, V)

In [5]:
class lstm_dataset(Dataset):

    def __init__(self, encoded_chunks):
        self.samples = []
        for chunk in encoded_chunks :
            X = chunk[:-1]
            y = chunk[1:]

            X = torch.tensor(X)
            y = torch.tensor(y)

            self.samples.append((X, y))

    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):
        context, target = self.samples[idx]
        return (
            torch.tensor(context, dtype=torch.long),
            torch.tensor(target, dtype=torch.long)
        )


def lstm_dataloader(batch_size, encoded_sentences) :
    dataset = lstm_dataset(encoded_sentences)
    train_loader = DataLoader(dataset, 
                              batch_size,
                              pin_memory=True,
                              shuffle=True)

    return train_loader

In [6]:
def training_loop(model, epochs, train_loader, optimizer, criterion):
    
    for epoch in range(epochs):
        print(f'STARTED EPOCH {epoch}')
        model.train()
        epoch_loss = 0.0
        for context,target in train_loader:
            batch_loss = 0

            context = context.to(device).long()
            target = target.to(device).long()
            
            optimizer.zero_grad()
            logits = model(context)  # shape = BxTxV

            logits = logits.reshape(-1, logits.size(-1)) #shape (B*t)xV
            target = target.reshape(-1) # shape (B*T)
            loss = criterion(logits, target)
            loss.backward()

            optimizer.step()

            batch_loss += loss.item()
            epoch_loss += loss.item() 
        print(f'EPOCH : {epoch} , Loss = {epoch_loss/len(train_loader)}')

In [9]:
dataloader = lstm_dataloader(32, encoded_chunks)
model_lstm = simpleLSTM(embedding_w, 5000, 50, 128, device, True)
model_lstm.to(device)

optimizer = Adam(
    model_lstm.parameters(),
    lr = 0.005
)

criterion = nn.CrossEntropyLoss()

In [10]:
training_loop(model_lstm, 100, dataloader, optimizer, criterion)

STARTED EPOCH 0


/tmp/ipykernel_37133/3178038646.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(context, dtype=torch.long),
/tmp/ipykernel_37133/3178038646.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(target, dtype=torch.long)


EPOCH : 0 , Loss = 6.797935103081368
STARTED EPOCH 1
EPOCH : 1 , Loss = 6.380908253386214
STARTED EPOCH 2
EPOCH : 2 , Loss = 6.150416942544886
STARTED EPOCH 3
EPOCH : 3 , Loss = 6.000597229519406
STARTED EPOCH 4
EPOCH : 4 , Loss = 5.899341230134707
STARTED EPOCH 5
EPOCH : 5 , Loss = 5.822746537182782
STARTED EPOCH 6
EPOCH : 6 , Loss = 5.759641102198008
STARTED EPOCH 7
EPOCH : 7 , Loss = 5.7050843174393115
STARTED EPOCH 8
EPOCH : 8 , Loss = 5.658329214920869
STARTED EPOCH 9
EPOCH : 9 , Loss = 5.617063204017845
STARTED EPOCH 10
EPOCH : 10 , Loss = 5.580954827489079
STARTED EPOCH 11
EPOCH : 11 , Loss = 5.548059241836135
STARTED EPOCH 12
EPOCH : 12 , Loss = 5.5176845486099655
STARTED EPOCH 13
EPOCH : 13 , Loss = 5.489377093959499
STARTED EPOCH 14
EPOCH : 14 , Loss = 5.468008947372437
STARTED EPOCH 15
EPOCH : 15 , Loss = 5.44543606268393
STARTED EPOCH 16
EPOCH : 16 , Loss = 5.423549409814783
STARTED EPOCH 17
EPOCH : 17 , Loss = 5.402949454333331
STARTED EPOCH 18
EPOCH : 18 , Loss = 5.385137

In [22]:
for i in range(10,11):
    chunk = encoded_chunks[i]
    context = torch.tensor(
        chunk[:-1],
        dtype=torch.long
    )
    context = context.unsqueeze(0)
    context = context.to(device)
    print(context.shape)
    logits = model_lstm(context) # shape 1,49,V
    pred = logits.argmax(dim = -1)
    pred = pred.squeeze(0)

    word_chunk = word_chunks[i][:-1]
    print(f"TRAIN : {word_chunk}")

    pred_chunk = [
        idx_word_dict[token.item()]
        for token in pred
    ]
       
    print(f"PRED : {pred_chunk}")


torch.Size([1, 49])
TRAIN : ['receiver', 'back', 'stroked', 'mustache', 'thinking', 'stupid', 'potter', 'unusual', 'name', 'sure', 'lots', 'people', 'called', 'potter', 'son', 'called', 'harry', 'come', 'think', 'even', 'sure', 'nephew', 'called', 'harry', 'never', 'even', 'seen', 'boy', 'might', 'harvey', 'harold', 'point', 'worrying', 'dursley', 'always', 'got', 'upset', 'mention', 'sister', 'blame', 'sister', 'like', 'people', 'cloaks', 'page', 'harry', 'potter', 'philosophers', 'stone']
PRED : ['<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', 'said', 'harry', 'said', 'enchantment', '<UNK>', 'lestrange', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', 'back', 'said', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', 'seen', '<UNK>', '<UNK>', '<UNK>', 'able', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', 'marge', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', 'harry', 'potter', 'deathly', 'stone', 'rowling']
